# Exercise 7

1. **Execute o código abaixo em um arquivo cpp da seguinte maneira**:
    
    * Na pasta ompenmp/notebooks/codigo_externo, crie um aquivo chamado: 
        * "exercise_7.cpp".

    * Clique duas vezes para abrí-lo no editor do VSCode.

    * Copie e cole o código abaixo dentro do arquivo e salve.

    * No terminal Linux, vá até a pasta do aquivo: 
        * cd ompenmp/notebooks/codigo_externo

    * Compile o aquivo:
        * g++ -fopenmp exercise_7.cpp -o exercise_7.exe

    * Execute o programa:
        ./exercise_7.exe

2. **Execute o código no notebook (single thread) e depois fora do notebook (número de threads == número de núcleos).**
    * Insira o código necessário para iniciar a contagem do tempo antes do paralelismo
    * Insira o código necessário para calcular o tempo corrigo após o término do paralelismo
    * Faça 3 execuções do código fora do notebook e, para cada uma, faça:
        * **Varie o valor de "n" e o número de threads**    
        * Imprima os resultados no console
        * Copie o tempo para a célula do tipo markdown que se encontra abaixo da célula de código
    * Compare e comente a variação nos tempos.

3. **Volte a este notebook**:
    * Na célula do tipo markdown abaixo da célula que contém o código a ser executado.

**Obs: dentro do notebook é executada apenas uma thread. Por isso, execute fora do notebook para ver o paralelismo.** 


**O código abaixo apresenta uma falha de segmentação.**

**Tente determinar o que está causando o erro e corrija.**

**Em seguida, execute conforme as instruções acima.**

In [1]:
#include "notebooks_reserved_code/openmp_config.h"
#include <omp.h>
#include <stdio.h>
#include <stdlib.h>

int main () 
{
	const int N=1048;
	int nthreads, tid, i, j;
	double a[N][N];

	/* Fork a team of threads with explicit variable scoping */
	#pragma omp parallel shared(nthreads) private(i,j,tid,a)
	{
		/* Obtain/print thread info */
		tid = omp_get_thread_num();
		if(tid == 0) 
		{
			nthreads = omp_get_num_threads();
			printf("Number of threads = %d\n", nthreads);
		}
		printf("Thread %d starting...\n", tid);

		/* Each thread works on its own private copy of the array */
		for (i=0; i<N; i++){
			for (j=0; j<N; j++){
			  a[i][j] = tid + i + j;
			}
		}

		/* For confirmation */
		printf("Thread %d done. Last element= %f\n",tid,a[N-1][N-1]);

	}  /* All threads join master thread and disband */

}
main()



input_line_7:1:10: fatal error: 'notebooks_reserved_code/openmp_config.h' file not found
#include "notebooks_reserved_code/openmp_config.h"
         ^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


Interpreter Error: 

## Resultados:

Cole os resultados das execuções.
| N    | Threads | Tempo (segundos)  |
|------|---------|-------------------|
| 500  | 1       | 0.001036          |
| 1000 | 4       | 0.006872          |
| 1000 | 8       | 0.005632          |

Com N=500 e uma thread, o tempo foi muito rápido devido ao tamanho reduzido do problema. Ao aumentar para N=1000, o tempo de execução cresceu, mesmo usando múltiplas threads, porque a quantidade de operações aumentou. Usar 8 threads trouxe uma leve melhora em relação a 4 threads, mas o ganho não foi grande, pois o overhead de criar e gerenciar várias threads começa a limitar a eficiência. Portanto, o paralelismo ajuda a acelerar a execução, mas com retornos decrescentes, e problemas maiores naturalmente demandam mais tempo de processamento.


Cole também o código corrigido.

O problema está na declaração da variável `a` como privada para cada thread, porém a variável é uma matriz grande alocada na stack, e isso gera stack overflow que provoca o erro de segmentação. A matriz `a` tem tamanho de 1048 x 1048 de double, o que da aproximadamente 8,7MB de memória, e como cada thread cria sua própria cópia da matriz na stack, ultrapassa o limite padrão de tamanho.


```c

#include "openmp_config.h"
#include <omp.h>
#include <stdio.h>
#include <stdlib.h>

int main()
{
    const int N = 1000;
    int nthreads, tid, i, j;
    double **a;

    double start_time, end_time;

    a = (double **)malloc(N * sizeof(double *));
    for (i = 0; i < N; i++)
    {
        a[i] = (double *)malloc(N * sizeof(double));
    }
    omp_set_num_threads(8); 

    start_time = omp_get_wtime();
/* Fork a team of threads with explicit variable scoping */
#pragma omp parallel shared(nthreads, a) private(i, j, tid)
    {
        /* Obtain/print thread info */
        tid = omp_get_thread_num();

        if (tid == 0)
        {
            nthreads = omp_get_num_threads();
            printf("Number of threads = %d\n", nthreads);
        }
        printf("Thread %d starting...\n", tid);

        /* Each thread works on its own private copy of the array */
        for (i = 0; i < N; i++)
        {
            for (j = 0; j < N; j++)
            {
                a[i][j] = tid + i + j;
            }
        }

        /* For confirmation */
        printf("Thread %d done. Last element= %f\n", tid, a[N - 1][N - 1]);

    } /* All threads join master thread and disband */

    end_time = omp_get_wtime();
    printf("Time taken = %f seconds\n", end_time - start_time);

    
    for (i = 0; i < N; i++)
    {
        free(a[i]);
    }
    free(a);

    return 0;
}

```

## Entrega:

**Salve este notebook com as suas respostas e poste como entrega da atividade no Canvas.**